# EDA — Solar Power Forecasting (24 governorates, 2005-2023)

This notebook is exploratory only: outlier checks, VIF analysis, correlation with the target, and a quick Random Forest importance check. It documents *why* the choices in `src/feature_engineering.py` were made.

The actual, reproducible pipeline (feature engineering -> split -> scaling -> training -> evaluation) lives in `src/` and is run with `python -m src.train`, **not** from this notebook.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))  # so `from src import ...` works from notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.ensemble import RandomForestRegressor

from src import config
from src.data_loader import load_raw_data

## 1. Load data

In [ ]:
df = load_raw_data()
print(df.shape)
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df.info()

## 2. Outlier check across all governorates

In [ ]:
features_outliers = ['G(i)', 'H_sun', 'T2m', 'WS10m', 'relative_humidity_2m',
                      'cloud_cover', 'wind_direction_10m', 'pressure_msl',
                      'precipitation', 'dew_point_2m', 'cloud_cover_low',
                      'cloud_cover_mid', 'cloud_cover_high', 'P']

plt.figure(figsize=(18, 10))
sns.boxplot(data=df[features_outliers], orient='h')
plt.title('Boxplots of Numerical Features')
plt.xlabel('Values')
plt.tight_layout()
plt.show()

In [ ]:
governorates = df['location'].unique()
n_cols = 3
n_rows = (len(features_outliers) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
axes = axes.flatten()

for i, feature in enumerate(features_outliers):
    ax = axes[i]
    for gov in governorates:
        gov_data = df[df['location'] == gov].iloc[:1000]
        ax.plot(gov_data[feature].values, label=gov, linewidth=0.8)
    ax.set_title(feature)
    ax.set_xlabel('First 1000 observations')

plt.tight_layout()
plt.show()

## 3. Wind direction — circular variable, encode as sin/cos

0 degrees and 359 degrees are neighbours in reality but look maximally far apart to a model reading raw degrees — the vertical jump artifacts in the plot above are this wraparound, not real volatility.

In [ ]:
df['wind_dir_sin'] = np.sin(np.radians(df['wind_direction_10m']))
df['wind_dir_cos'] = np.cos(np.radians(df['wind_direction_10m']))
df = df.drop(columns=['wind_direction_10m'])

## 4. Cloud cover redundancy — VIF analysis

`cloud_cover` (total) is typically derived from the low/mid/high layers in weather datasets. Checking VIF (not just pairwise correlation, since multicollinearity from a *combination* of features can hide behind weak individual correlations) confirms this.

In [ ]:
cloud_cols = ['cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high']
print(df[cloud_cols].corr())

In [ ]:
X = df[cloud_cols].dropna()
vif_data = pd.DataFrame()
vif_data['feature'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif_data)

`cloud_cover` came out at **VIF ≈ 15** (≈93% explained by the other three combined) — drop it and re-check the remaining three.

In [ ]:
df = df.drop(columns=['cloud_cover'])

cloud_cols_reduced = ['cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high']
X = df[cloud_cols_reduced].dropna()
vif_data = pd.DataFrame()
vif_data['feature'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif_data)

All three now sit close to VIF ≈ 1 — no meaningful collinearity remains. Keep `cloud_cover_low/mid/high`, drop `cloud_cover`.

## 5. Correlation of every feature with the target P

Note: Pearson correlation only captures *linear* relationships. `hour` and `month` will look weak here even though they matter a lot — see the cyclical encoding + Random Forest importance sections below for why.

In [ ]:
corr_with_target = df.corr(numeric_only=True)['P'].drop('P').sort_values()

plt.figure(figsize=(8, 10))
colors = ['crimson' if v < 0 else 'steelblue' for v in corr_with_target]
plt.barh(corr_with_target.index, corr_with_target.values, color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Correlation of each feature with P')
plt.xlabel('Pearson correlation coefficient')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Cyclical encoding of hour / month / day

In [ ]:
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['month_sin'] = np.sin(2 * np.pi * (df['month'] - 1) / 12)
df['month_cos'] = np.cos(2 * np.pi * (df['month'] - 1) / 12)
df['day_sin'] = np.sin(2 * np.pi * df['day'] / 31)
df['day_cos'] = np.cos(2 * np.pi * df['day'] / 31)
df = df.drop(columns=['hour', 'month', 'day'])
df.columns.tolist()

## 7. Random Forest feature importance (captures nonlinear relationships)

Complementary to Pearson correlation — this is where `hour_sin`/`hour_cos` should show up as meaningful, unlike the raw `hour` column above.

In [ ]:
target = 'P'
feature_cols = [c for c in df.columns if c not in [target, 'location']]

data = df[feature_cols + [target]].dropna()
X = data[feature_cols]
y = data[target]

rf = RandomForestRegressor(n_estimators=300, max_depth=None, random_state=42, n_jobs=-1)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)

In [ ]:
plt.figure(figsize=(8, 10))
importances.sort_values().plot(kind='barh', color='seagreen')
plt.title('Random Forest Feature Importance (predicting P)')
plt.xlabel('Importance (Gini-based)')
plt.tight_layout()
plt.show()

## Conclusions carried into `src/feature_engineering.py`

- Drop `cloud_cover` (redundant with low/mid/high, VIF ≈ 15)
- Drop `wind_direction_10m`, `hour`, `month`, `day` — replaced by sin/cos pairs
- Drop `location` in the modeling pipeline — `latitude`/`longitude` already encode each governorate's position numerically
- Do not drop features on Pearson correlation alone (see `hour`) — cross-check with Random Forest importance, which captures nonlinear/cyclical effects